# 🧹 Lekcja 3: Wielkie sprzątanie — brakujące dane (`NaN`), logika Boole'a i tajemnice `data_description.txt`

W idealnym świecie każda tabela z danymi ma wypełnione wszystkie kratki od A do Z. W prawdziwym świecie — czy to w banku, szpitalu, czy w biurze nieruchomości — **w danych zawsze są dziury**.

W Pythonie i bibliotece `pandas` każda taka pusta kratka nazywa się **`NaN`** (skrót od angielskiego ***Not a Number*** — dosłownie: *„To nie jest liczba / Brak wartości”*).

---

### ⚠️ Dlaczego modele Machine Learningu nienawidzą `NaN`? (Wyjaśnienie matematyczne)

Wyobraź sobie, że model Machine Learningu to pod maską duże równanie matematyczne, które liczy cenę domu, dodając do siebie punkty za kolejne cechy:

$$\text{Cena} = (100 \$ \times \text{Metraż}) + (500 \$ \times \text{DługośćPłotu}) + (15\,000 \$ \times \text{Jakość})$$

Co się stanie, jeśli w danym domu nie znamy długości płotu i w tabeli widnieje tam **`NaN`**?  
W matematyce komputerowej obowiązuje twarda zasada — **`NaN` zaraża każde działanie**:
* $500 \times \text{NaN} = \text{NaN}$
* $150\,000 + \text{NaN} + 90\,000 = \text{NaN}$

**Jedna pusta kratka sprawia, że całe równanie wybucha**, a większość algorytmów Machine Learningu od razu zatrzymuje pracę i wyrzuca czerwony błąd: `ValueError: Input contains NaN`!  
Dlatego zanim zaczniemy uczyć jakikolwiek model, musimy znaleźć wszystkie dziury `NaN`, zrozumieć, **skąd się wzięły**, i mądrze je załatać.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Wczytujemy dane z folderu ../data/
train = pd.read_csv('../data/train.csv')

# 2. Liczymy, ile brakujących wartości (NaN) jest w każdej z 81 kolumn
wszystkie_braki = train.isnull().sum()

# 3. Zostawiamy tylko te kolumny, gdzie liczba braków jest większa od zera (> 0)
#    i od razu ustawiamy je w kolejności od największej do najmniejszej (.sort_values)
tylko_braki = wszystkie_braki[wszystkie_braki > 0].sort_values(ascending=False)

# Wypisujemy tabelkę z liczbą braków
print('Kolumny z brakującymi danymi (spośród 1460 domów):')
print(tylko_braki)

# 4. Rysujemy wykres słupkowy, żeby zobaczyć skalę problemu własnymi oczami
plt.figure(figsize=(12, 6))
plt.bar(tylko_braki.index, tylko_braki.values, color='coral', edgecolor='black')

# Linia pokazująca łączną liczbę wszystkich domów w bazie (1460)
plt.axhline(1460, color='red', linestyle='--', linewidth=1.5, label='Wszystkie domy w bazie (1460)')

plt.title('Liczba brakujących wartości (NaN) w poszczególnych kolumnach')
plt.xlabel('Nazwa kolumny')
plt.ylabel('Liczba pustych komórek (NaN)')
plt.xticks(rotation=45, ha='right')  # Obracamy napisy na dole o 45 stopni, żeby na siebie nie nachodziły
plt.legend()
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

### 🔬 Co dokładnie dzieje się pod maską w `train.isnull().sum()`? (Rozbiórka matematyczno-programistyczna)

Linijka `wszystkie_braki = train.isnull().sum()` wygląda krótko, ale komputer wykonuje w niej **dwa wielkie kroki matematyczne** jeden po drugim. Nazywamy to w Pythonie **łańcuchem metod (*Method Chaining*)** — zamiast pisać dwie osobne linijki, łączymy kolejne polecenia kropkami jak wagony pociągu.

Zobaczmy na małym wycinku 3 domów, co dokładnie robi komputer:

#### KROK 1: `train.isnull()` — Detektor pustych kratek
Metoda `.isnull()` (dosłownie: *„czy jest puste?”*) zagląda do każdej kratki w tabeli i zadaje pytanie: **„Czy tutaj jest dziura `NaN`?”**.  
Zamiast cen, tekstów i liczb zwraca zupełnie nową tabelę wypełnioną tylko dwoma słowami z tzw. **logiki Boole'a (logiki zero-jedynkowej)**:
* **`True` (Prawda)** — TAK, kratka jest pusta (`NaN`)!
* **`False` (Fałsz)** — NIE, w kratce jest normalna wartość.

| Dom | `LotFrontage` (Oryginał) | $\rightarrow$ `LotFrontage` po `.isnull()` | `PoolQC` (Oryginał) | $\rightarrow$ `PoolQC` po `.isnull()` |
| :--- | :--- | :--- | :--- | :--- |
| **Dom 1** | `65.0` | `False` (nie jest puste) | `NaN` | **`True`** (jest puste!) |
| **Dom 2** | `80.0` | `False` (nie jest puste) | `NaN` | **`True`** (jest puste!) |
| **Dom 3** | `NaN` | **`True`** (jest puste!) | `'Ex'` | `False` (nie jest puste) |

---

#### KROK 2: `.sum()` — Jak komputer może „dodawać” słowa `True` i `False`?!
Na końcu dopisaliśmy `.sum()`, czyli *„dodaj do siebie wszystkie wiersze w każdej kolumnie”*. Ale zaraz — jak można dodać do siebie słowa `False + False + True`?!

Tu kryje się jeden z najciekawszych sekretów informatyki i Pythona:
> 💡 **Dla procesora komputera słowo `True` to po prostu liczba `1`, a słowo `False` to liczba `0`!**  
> (Możesz to nawet sprawdzić sam w Pythonie: jeśli wpiszesz `True + True + False`, Python policzy $1 + 1 + 0$ i wyświetli wynik **`2`**!).

Kiedy więc wywołujemy `.sum()` na tabeli pełnej `True` i `False`, komputer w każdej kolumnie zamienia `True` na `1`, a `False` na `0` i dodaje je w pionie:
* W kolumnie `LotFrontage` z naszej tabelki wyżej: `False + False + True` = $0 + 0 + 1 =$ **`1 brak`** (a w całej tabeli 1460 domów uzbierało się tak **`259`** jedynek).
* W kolumnie `PoolQC`: `True + True + False` = $1 + 1 + 0 =$ **`2 braki`** (a w całej tabeli aż **`1453`** jedynki!).

---

#### Pozostałe nowe elementy składni z powyższej komórki:
* **`wszystkie_braki[wszystkie_braki > 0]`** — spośród 81 kolumn aż 62 kolumny miały wynik `0` (zero braków). Nawias kwadratowy z warunkiem `> 0` odfiltrował zera i zostawił tylko **19 kolumn**, w których naprawdę czegoś brakuje.
* **`.sort_values(ascending=False)`** — sortuje wyniki według wartości. Słowo `ascending` oznacza „rosnąco” (od najmniejszej do największej). Ustawiając `ascending=False` (czyli *„czy rosnąco? Fałsz!”*), każemy mu posortować wyniki **malejąco** — od rekordzisty (`PoolQC`: 1453 braki) do najmniejszego problemu (`Electrical`: 1 brak).
* **`plt.bar(tylko_braki.index, tylko_braki.values)`** — rysuje wykres słupkowy (`bar` = słupek):
  * `.index` to etykiety (nazwy naszych 19 kolumn, które trafiają na oś poziomą X),
  * `.values` to przypisane do nich liczby braków (wysokości słupków na osi pionowej Y).
* **`plt.xticks(rotation=45, ha='right')`** — `xticks` to napisy pod osią X. Ponieważ nazwy 19 kolumn są długie, obracamy je o 45 stopni (`rotation=45`) i wyrównujemy do prawej (`ha='right'`), żeby nie nachodziły na siebie.

---
### 🕵️‍♂️ Zagadka detektywistyczna: Dlaczego w `PoolQC` brakuje aż 99.5% danych?!

Spójrz na pierwszy słupek na naszym wykresie: **`PoolQC` ma aż 1453 puste komórki na 1460 domów (99.5% braków!)**.  
Początkujący adept Data Science często mówi w tym momencie: *„Ojej, ta kolumna jest kompletnie zepsuta, ktoś zapomniał wpisać dane dla 1453 domów! Wyrzućmy ją do kosza!”*.

**To byłby ogromny błąd!** Zobaczmy, co mówi nasz słownik — plik **`../data/data_description.txt`** — gdy poszukamy w nim opisów dla `PoolQC`, `GarageType`, `Alley` czy `FireplaceQu`:

```text
PoolQC: Pool quality
       Ex   Excellent
       Gd   Good
       TA   Average/Typical
       Fa   Fair
       NA   No Pool          <--- SPÓJRZ TUTAJ!

GarageType: Garage location
       Attchd   Attached to home
       Detchd   Detached from home
       ...
       NA       No Garage    <--- I TUTAJ!

Alley: Type of alley access to property
       Grvl     Gravel
       Pave     Paved
       NA       No alley access
```

#### 💡 Co tu się wydarzyło?! („Pozorny brak danych” kontra „Prawdziwy brak informacji”)

1. **Skąd wziął się `NaN` w basenach i garażach?**  
   Kiedy urzędnik w USA wypełniał ankietę o domu, który **nie miał basenu**, wpisywał w rubrykę `PoolQC` dwie litery: **`NA`** (od *Not Applicable* — „nie dotyczy / brak basenu”).  
   Kiedy my w Pythonie użyliśmy funkcji `pd.read_csv('../data/train.csv')`, biblioteka `pandas` zobaczyła w pliku tekstowym litery `NA` i automatycznie przetłumaczyła je sobie jako *Not Available* (**brak danych `NaN`**)!

2. **Czym więc różnią się dwa rodzaje braków w naszej tabeli?**

| Rodzaj braku | Przykłady kolumn w naszych danych | Co ten brak naprawdę oznacza w świecie rzeczywistym? | Jak musimy go naprawić (uzupełnić)? |
| :--- | :--- | :--- | :--- |
| **1. Pozorny brak danych** *(Cecha nie istnieje w tym domu)* | • **Tekstowe (15 kolumn):** `PoolQC` (basen), `MiscFeature` (szopa/kort), `Alley` (alejka), `Fence` (płot), `FireplaceQu` (kominek), 4 kolumny `Garage...` (81 domów bez garażu), 5 kolumn `Bsmt...` (37 domów bez piwnicy), `MasVnrType` (okładzina kamienna)<br>• **Liczbowe (2 kolumny):** `MasVnrArea` (powierzchnia okładziny), `GarageYrBlt` (rok budowy garażu) | **To NIE jest zgubiona informacja!** Wiemy na 100%, że ten dom po prostu **nie posiada basenu, garażu albo piwnicy**. | • W kolumnach **tekstowych** wpisujemy słowo **`'Brak'`** (tworzymy pełnoprawną kategorię „Brak basenu”).<br>• W kolumnach **liczbowych** wpisujemy liczbę **`0`** (skoro nie ma okładziny kamiennej, jej powierzchnia to `0` $m^2$). |
| **2. Prawdziwy brak informacji** *(Ktoś naprawdę nie wpisał danych!)* | • **`LotFrontage` (259 braków — liczba):** szerokość frontu działki wzdłuż ulicy (w stopach).<br>• **`Electrical` (1 brak — tekst):** typ instalacji elektrycznej w domu. | **Każdy dom styka się jakoś z ulicą i ma jakąś instalację elektryczną!** Tutaj ankieter naprawdę zapomniał zmierzyć płot od strony ulicy w 259 domach, a w 1 domu ominął rubrykę z prądem. | Nie wolno tu wpisać `0` ani `'Brak'`! Musimy **statystycznie odgadnąć (imputować)** najbardziej prawdopodobną wartość:<br>• dla **liczb (`LotFrontage`)** wstawiamy **MEDIANĘ (`69.0` stóp)**,<br>• dla **tekstu (`Electrical`)** wstawiamy **DOMINANTĘ / MODĘ (`'SBrkr'`)**, czyli najpopularniejszy typ instalacji. |

---

### 🧮 Matematyka łatania dziur: Dlaczego `LotFrontage` uzupełniamy MEDIANĄ (`69.0`), a `Electrical` DOMINANTĄ (`'SBrkr'`)?

1. **Dlaczego w `LotFrontage` (szerokość działki przy ulicy) NIE wpiszemy `0`?**  
   Gdybyśmy wstawili `0`, powiedzielibyśmy modelowi: *„Te 259 domów ma 0 metrów styku z ulicą (lewitują w powietrzu!)”*. Model zgłupieje.
2. **Dlaczego więc wstawiamy MEDIANĘ (`69.0` stóp), a nie średnią (`70.05` stóp)?**  
   Pamiętasz z Lekcji 1 przypowieść o **miliarderze w barze**? W kolumnie `LotFrontage` też jest kilku „miliarderów” — ogromne rancza z frontem działki o długości ponad 300 stóp, które zawyżają średnią. **Mediana (`69.0` stóp)** to wartość środkowego, najbardziej typowego domu, odporna na skrajne przypadki.
3. **Co to jest DOMINANTA / MODA (`.mode()`) przy uzupełnianiu tekstu (`Electrical`)?**  
   W kolumnie `Electrical` mamy słowa (kategorie): `'SBrkr'` (standardowe bezpieczniki), `'FuseA'`, `'FuseF'` itd. Z tekstu nie da się policzyć średniej ani mediany!  
   Zamiast tego sprawdzamy, **które słowo występuje najczęściej** (statystycy nazywają to **modą** lub **dominantą** — po angielsku `mode`):
   * `'SBrkr'` występuje w **1334 domach na 1460 (91.4% wszystkich domów!)**.
   * Skoro ponad 9 na 10 domów ma instalację `'SBrkr'`, to uzupełnienie tego jednego brakującego wiersza wartością `'SBrkr'` ma ponad 91% szans na bycie strzałem w dziesiątkę!

In [ ]:
# Zapisujemy kopię kolumny LotFrontage PRZED uzupełnieniem, żeby porównać ją potem na wykresie
front_przed = train['LotFrontage'].dropna().copy()

# --- GRUPA 1: Pozorne braki TEKSTOWE -> zamieniamy NaN na słowo 'Brak' ---
kolumny_tekstowe_brak = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'
]

for kolumna in kolumny_tekstowe_brak:
    train[kolumna] = train[kolumna].fillna('Brak')

# --- GRUPA 2: Pozorne braki LICZBOWE -> zamieniamy NaN na liczbę 0 ---
train['MasVnrArea'] = train['MasVnrArea'].fillna(0)
train['GarageYrBlt'] = train['GarageYrBlt'].fillna(0)

# --- GRUPA 3: Prawdziwe braki informacji -> Mediana (dla liczb) i Dominanta/Moda (dla tekstu) ---
mediana_frontu = train['LotFrontage'].median()
train['LotFrontage'] = train['LotFrontage'].fillna(mediana_frontu)

najczestsza_elektryka = train['Electrical'].mode()[0]
train['Electrical'] = train['Electrical'].fillna(najczestsza_elektryka)

# SPRAWDZENIE: Ile łącznie pustych kratek (NaN) zostało teraz w całej tabeli train?
pozostale_braki = train.isnull().sum().sum()
print(f'Uzupełniono LotFrontage medianą: {mediana_frontu} stóp')
print(f'Uzupełniono Electrical najczęstszą wartością: {najczestsza_elektryka}')
print(f'Łączna liczba braków (NaN) w całej tabeli train PO czyszczeniu: {pozostale_braki}')

# WIZUALIZACJA: Zobaczmy na dwóch wykresach obok siebie efekt naszej pracy!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Wykres lewy: Co wpisaliśmy do kolumny PoolQC (Baseny)?
policzone_baseny = train['PoolQC'].value_counts()
ax1.bar(policzone_baseny.index, policzone_baseny.values, color=['lightgray', 'gold', 'skyblue', 'lightgreen'], edgecolor='black')
ax1.set_title('1. Kolumna PoolQC po zamianie NaN na „Brak”')
ax1.set_xlabel('Kategoria w kolumnie PoolQC')
ax1.set_ylabel('Liczba domów')
for i, wartosc in enumerate(policzone_baseny.values):
    ax1.text(i, wartosc + 20, str(wartosc), ha='center', fontweight='bold')
ax1.set_ylim(0, 1600)

# Wykres prawy: Rozkład LotFrontage PRZED i PO uzupełnieniu 259 braków medianą (69.0)
ax2.hist(train['LotFrontage'], bins=40, color='lightgreen', edgecolor='black', alpha=0.7, label='PO uzupełnieniu (z 259 medianami = 69.0)')
ax2.hist(front_przed, bins=40, color='royalblue', edgecolor='black', alpha=0.7, label='PRZED uzupełnieniem (bez 259 domów)')
ax2.axvline(mediana_frontu, color='red', linestyle='--', linewidth=2, label=f'Mediana ({mediana_frontu})')
ax2.set_title('2. Rozkład LotFrontage przed i po wstawieniu mediany')
ax2.set_xlabel('Szerokość frontu działki (LotFrontage w stopach)')
ax2.set_ylabel('Liczba domów')
ax2.legend()

plt.tight_layout()
plt.show()

### 🔍 Rozbiórka nowego kodu i wykresów na czynniki pierwsze

Spójrz najpierw na **prawy wykres** powyżej! Widzisz ten wysoki **zielony słupek wystający dokładnie na czerwonej przerywanej linii (`69.0`)**?  
Niebieski histogram to domy, które miały zmierzony front działki od początku. Zielony słupek pośrodku wystrzelił w górę o dokładnie **259 domów**, ponieważ wszystkim 259 domom, w których brakowało pomiaru, przypisaliśmy bezpieczną, środkową wartość **`69.0` stóp**!

Rozszyfrujmy teraz każdą nową konstrukcję Pythona, której użyliśmy:

#### 1. Metoda `.fillna(...)` — Szpachla na dziury w danych
Nazwa `fillna` to skrót od angielskiego ***Fill NA*** (*„Wypełnij braki `NaN`”*):
* `train['MasVnrArea'].fillna(0)` — szuka w kolumnie `MasVnrArea` wszystkich pustych kratek `NaN` i wstawia w ich miejsce liczbę `0` (reszty domów nie dotyka!).
* **Dlaczego piszemy `train['MasVnrArea'] = train['MasVnrArea'].fillna(0)`?**  
  Ponieważ `.fillna(0)` domyślnie nie zmienia oryginalnej tabeli w miejscu, tylko tworzy jej poprawioną kopię. Pisząc `train['MasVnrArea'] = ...`, nadpisujemy starą, dziurawą kolumnę nową, naprawioną wersją.

#### 2. Lista `[...]` oraz pętla `for kolumna in kolumny_tekstowe_brak:` — Taśma produkcyjna
Mieliśmy aż 15 kolumn tekstowych do uzupełnienia słowem `'Brak'`. Gdybyśmy pisali `.fillna('Brak')` 15 razy linijka pod linijką, nasz kod byłby nudny i długi. Zamiast tego:
1. **Stworzyliśmy listę `kolumny_tekstowe_brak = ['PoolQC', 'MiscFeature', ...]`** — w Pythonie pojedynczy nawias kwadratowy przypisany do zmiennej to **lista** (jak lista zakupów na kartce).
2. **Użyliśmy pętli `for` (dla każdego elementu z listy):**
   ```python
   for kolumna in kolumny_tekstowe_brak:
       train[kolumna] = train[kolumna].fillna('Brak')
   ```
   * Jak to czytać po polsku? *„**Dla każdej** nazwy (którą chwilowo nazwiemy `kolumna`) znajdującej się **w** naszej liście `kolumny_tekstowe_brak`, wykonaj wciętą linijkę kodu pod spodem”*.
   * **Dwukropek `:`** na końcu pierwszej linijki oraz **wcięcie (4 spacje / Tab)** w drugiej linijce są w Pythonie obowiązkowe! Wcięcie pokazuje komputerowi, które linijki kodu mają się powtarzać w kółko na taśmie produkcyjnej.

#### 3. Zagadka nawiasu `[0]` w `train['Electrical'].mode()[0]`
Dlaczego przy średniej piszemy po prostu `.mean()`, przy medianie `.median()`, a przy dominancie (modzie) musimy dopisać na końcu **`.mode()[0]`**?
* Średnia i mediana to zawsze **jedna konkretna liczba**.
* A co z najczęstszą wartością (`.mode()`)? Wyobraź sobie ankietę, w której 50 osób wybrało kolor niebieski i dokładnie 50 osób wybrało kolor czerwony. Mamy **remis na pierwszym miejscu**!  
  Dlatego twórcy biblioteki `pandas` zdecydowali, że `.mode()` zawsze zwraca **listę zwycięzców** (nawet jeśli zwycięzca jest tylko jeden).
* Dopisując na końcu **`[0]`**, mówimy Pythonowi: *„Wyciągnij z listy zwycięzców **pierwszy element**”* (pamiętaj: **w Pythonie numerowanie pozycji na liście zawsze zaczyna się od `0`**, a nie od `1`!).

#### 4. Dlaczego na końcu napisaliśmy podwójne `.sum().sum()` w `train.isnull().sum().sum()`?
* Pierwsze `.sum()` sumuje braki w pionie dla każdej z 81 kolumn osobno (daje listę 81 wyników).
* Drugie `.sum()` sumuje te 81 wyników ze sobą w jedną ostateczną liczbę — sprawdzając, czy w całej tabeli 1460 $\times$ 81 zostało dokładnie **`0`** pustych kratek!

In [ ]:
# 🎯 ĆWICZENIE 1: Znajdźmy i naprawmy pozorne braki w pliku test.csv
#
# 1. Wczytaj plik '../data/test.csv' do zmiennej `test` za pomocą pd.read_csv('../data/test.csv')
# 2. Uzupełnij puste miejsca w kolumnie test['PoolQC'] słowem 'Brak' za pomocą .fillna('Brak')
# 3. Uzupełnij puste miejsca w kolumnie test['GarageYrBlt'] liczbą 0 za pomocą .fillna(0)
# 4. Wypisz liczbę pozostałych braków w tych dwóch kolumnach (.isnull().sum())

# Miejsce na Twój kod:


In [ ]:
# 🎯 ĆWICZENIE 2: Prawdziwy brak danych w test.csv — uzupełnianie medianą (liczby) oraz dominantą (tekst)
# W tabeli `test` brakuje 227 pomiarów frontu działki ('LotFrontage' — liczba)
# oraz 4 wpisów o strefie zabudowy ('MSZoning' — tekst, np. 'RL' = strefa mieszkalna).
#
# 1. Oblicz medianę kolumny test['LotFrontage'] (.median()) i uzupełnij nią braki w test['LotFrontage']
# 2. Znajdź najczęstszą wartość w kolumnie test['MSZoning'] za pomocą .mode()[0] i uzupełnij nią braki w test['MSZoning']
# 3. Sprawdź za pomocą print() i .isnull().sum(), czy w obu kolumnach liczba braków spadła do 0!

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Jak działa `.isnull().sum()`:** Metoda `.isnull()` zamienia tabelę na wartości logiczne `True` (jest pusta kratka `NaN`) oraz `False` (jest wartość), a ponieważ dla procesora `True = 1` i `False = 0`, metoda `.sum()` zlicza dokładną liczbę braków w każdej kolumnie.
2. **Pozorny brak kontra prawdziwa dziura w danych:** Zaglądając do `data_description.txt`, odkryliśmy, że w większości kolumn (np. `PoolQC`, `GarageType`) `NaN` powstał z liter `NA` oznaczających fizyczny brak basenu lub garażu — takie miejsca wypełniamy słowem `'Brak'` (dla tekstu) lub liczbą `0` (dla powierzchni/roku).
3. **Statystyczne łatanie prawdziwych dziur (`.fillna()`):** Tam, gdzie cecha na pewno istnieje, ale ktoś zapomniał ją wpisać, liczby uzupełniamy odporną na skrajności **medianą (`.median()`)**, a tekst najczęściej występującą wartością, czyli **dominantą/modą (`.mode()[0]`)**.